# Migration Lab: Traditional NSGA-II to Framework / 传统 NSGA-II 到框架

Goal: migrate a hand-written multi-objective NSGA-II script into NSGABlack architecture.
目标：把手写多目标 NSGA-II 脚本迁移为 NSGABlack 架构。


In [ ]:
from pathlib import Path
import sys

cur = Path.cwd().resolve()
for _ in range(12):
    if (cur / 'pyproject.toml').exists() and (cur / '__init__.py').exists():
        if str(cur.parent) not in sys.path:
            sys.path.insert(0, str(cur.parent))
        print('bootstrap ok:', cur)
        break
    cur = cur.parent


## Step 0: Baseline responsibilities / 第 0 步：传统脚本职责
Traditional NSGA-II scripts often mix all algorithm mechanics in one file:
传统 NSGA-II 脚本通常把所有算法机制混在一个文件里：
- objective functions / 目标函数
- rank + crowding logic / 分层排序 + 拥挤距离
- selection/crossover/mutation / 选择/交叉/变异
- environmental selection / 环境选择


In [ ]:
from pathlib import Path
print('baseline script:', Path('00_baseline.py').resolve())
print('framework script:', Path('02_framework_final.py').resolve())


## Step 1: Move objectives to Problem layer / 第 1 步：把目标迁入 Problem 层
Define a `BlackBoxProblem` subclass that only expresses objective semantics.
定义 `BlackBoxProblem` 子类，只表达目标语义。


In [ ]:
import numpy as np
from nsgablack.core.base import BlackBoxProblem

class BiObjectiveShiftedSphere(BlackBoxProblem):
    def __init__(self, dimension=12, low=-5.0, high=5.0):
        bounds = {f'x{i}': (low, high) for i in range(dimension)}
        super().__init__(name='BiObjectiveShiftedSphere', dimension=dimension, bounds=bounds, objectives=['f1', 'f2'])

    def evaluate(self, x):
        arr = np.asarray(x, dtype=float)
        return np.array([np.sum(arr ** 2), np.sum((arr - 2.0) ** 2)], dtype=float)


## Step 2: Move operators to pipeline / 第 2 步：把算子迁入 Pipeline
Use `RepresentationPipeline` for initializer/mutator/repair.
使用 `RepresentationPipeline` 承载初始化/变异/修复。


In [ ]:
from nsgablack.representation import RepresentationPipeline
from nsgablack.representation.continuous import UniformInitializer, GaussianMutation, ClipRepair

pipeline = RepresentationPipeline(
    initializer=UniformInitializer(low=-5.0, high=5.0),
    mutator=GaussianMutation(sigma=0.25, low=-5.0, high=5.0),
    repair=ClipRepair(low=-5.0, high=5.0),
)
pipeline


## Step 3: Move NSGA-II process to adapter / 第 3 步：把 NSGA-II 过程迁入 Adapter
Use `NSGA2Adapter` instead of manually coding rank/crowding/select/update loops.
使用 `NSGA2Adapter` 替代手写的分层/拥挤/选择/更新主循环。


In [ ]:
from nsgablack.adapters import NSGA2Adapter, NSGA2Config

adapter = NSGA2Adapter(NSGA2Config(population_size=80, offspring_size=80, crossover_rate=0.9))
adapter


## Step 4: Assemble and run / 第 4 步：组装并运行
`ComposableSolver` provides the runtime lifecycle shell.
`ComposableSolver` 提供运行期生命周期外壳。


In [ ]:
from nsgablack.core.composable_solver import ComposableSolver
from nsgablack.utils.performance.fast_non_dominated_sort import FastNonDominatedSort

problem = BiObjectiveShiftedSphere(dimension=12, low=-5.0, high=5.0)
solver = ComposableSolver(problem=problem, adapter=adapter, representation_pipeline=pipeline)
solver.set_max_steps(80)
solver.set_random_seed(11)
result = solver.run()
obj = adapter.objectives
vio = adapter.violations
fronts, _ = FastNonDominatedSort.sort(obj, vio)
print(result)
print('front0 size:', len(fronts[0]) if fronts else 0)


## Step 5: Compare and verify / 第 5 步：对比与验证
- Baseline file / 传统文件: `00_baseline.py`
- Framework file / 框架文件: `02_framework_final.py`
- Mapping doc / 映射文档: `03_diff.md`

Expected outcome: comparable Pareto-front behavior with clearer architecture boundaries.
预期结果：Pareto 前沿行为与传统版可对照，但架构边界更清晰。
